# Early-training success rate vs 65% criterion

Célia asked to check whether **FL_M01569519**, **MbL_M01569517** and **MbR_M01569518**
reached **65% success in their last two sessions** of early training. Each row of her
tables is one day (pooling the Bonsai sessions listed for it), so this reuses the same
`(training_day, session, min_trial, exclude_on)` spec shape as `TRAINING_DETAIL` and the
same `training_spec` / `filter_trials` helpers.

Output: success rate per day, the three mice overlaid, with a dashed line at 65%, plus a
per-mouse table of the last-two-day success rates so the "reached 65%?" question is
answered as a number.

Note (per Célia): FL's `2026-04-29T170501Z` is set `exclude_on=True` as a safety net -
she wasn't certain there were no ON (cue) trials that day, so any are dropped.

In [ ]:
# Setup
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_conduit.qc import (
    qc_datastructure, training_spec, training_session_names, filter_trials,
)

# These are late-April training sessions, which live under the Training tree, so load
# from the common BonsaiOutput parent (depth=3, <tree>/<mouseID>/<day>/<session>) exactly
# like Test_success_rate.ipynb; include= still restricts to the sessions listed below.
ROOT = pathlib.Path('/media/sepi/Elements1/PathIntegrationProtocol/BonsaiOutput')

CRITERION = 0.65   # the success-rate criterion Célia is checking against

In [ ]:
# One row per day (repeat the day number to pool that day's sessions), transcribed from
# Célia's TSVs. Tuple = (training_day, session, min_trial, exclude_on).
CRITERION_DETAIL: dict[str, list[tuple[int, str, int, bool]]] = {
    'FL_M01569519': [
        (1, '2026-04-23T151952Z', 11, False),
        (1, '2026-04-23T152710Z', 11, False),
        (1, '2026-04-23T153735Z',  6, False),
        (2, '2026-04-25T194933Z', 11, False),
        (2, '2026-04-25T200011Z', 11, False),
        (2, '2026-04-25T200705Z', 11, False),
        (2, '2026-04-25T201616Z', 11, False),
        (2, '2026-04-25T202109Z',  6, False),
        (3, '2026-04-28T172539Z', 21, False),
        (3, '2026-04-28T173816Z',  6, False),
        (3, '2026-04-28T174249Z',  6, False),
        (3, '2026-04-28T174922Z',  2, False),
        (4, '2026-04-29T170501Z', 11, True),    # exclude_on=True (Célia: in case there were ON trials)
        (4, '2026-04-29T171313Z', 11, False),
        (4, '2026-04-29T172239Z', 11, False),
        (4, '2026-04-29T174637Z',  2, False),
    ],
    'MbL_M01569517': [
        (1, '2026-04-25T161723Z', 11, False),
        (1, '2026-04-25T163456Z', 11, False),
        (2, '2026-04-26T144900Z', 11, False),
        (3, '2026-04-27T132909Z', 11, False),
        (3, '2026-04-27T134004Z', 11, False),
        (3, '2026-04-27T135152Z', 11, False),
        (4, '2026-04-28T104553Z', 11, False),
        (4, '2026-04-28T105504Z', 11, False),
        (5, '2026-04-29T124453Z', 11, False),
        (5, '2026-04-29T125524Z',  4, False),
        (5, '2026-04-29T130235Z',  4, False),
        (5, '2026-04-29T130649Z',  4, False),
        (6, '2026-05-07T093236Z', 16, False),   # after shaping
        (7, '2026-05-08T111319Z', 16, False),   # last 2 days (6 & 7) are the post-shaping ones
    ],
    'MbR_M01569518': [
        (1, '2026-04-25T144913Z', 11, False),
        (1, '2026-04-25T150252Z', 11, False),
        (1, '2026-04-25T151326Z', 11, False),
        (2, '2026-04-26T163755Z', 11, False),
        (2, '2026-04-26T164301Z', 11, False),
        (2, '2026-04-26T165604Z', 11, False),
        (2, '2026-04-26T170917Z', 11, False),
        (3, '2026-04-27T141618Z', 11, False),
        (3, '2026-04-27T142320Z', 11, False),
        (4, '2026-04-28T100411Z', 11, False),
        (5, '2026-04-29T113619Z', 11, False),
        (5, '2026-04-29T114758Z',  4, False),
        (5, '2026-04-29T115844Z',  4, False),
        (5, '2026-04-29T120528Z',  4, False),
    ],
}

In [ ]:
# Helpers.
DEFAULT_MOUSE_COLOURS = {
    'FL_M01569519': '#950041', 'MbL_M01569517': '#004cd9', 'MbR_M01569518': '#2e7ebc',
}


def make_outcome_df(trials, outcome_filters, *, group_cols, outcome_col='outcome'):
    """Per-group outcome counts and proportions (n_<name>, prop_<name>)."""
    group_cols = list(group_cols)
    out = trials.groupby(group_cols).size().rename('n_total').reset_index()
    for name, outcomes in outcome_filters.items():
        counts = (trials[trials[outcome_col].isin(outcomes)]
                  .groupby(group_cols).size().rename(f'n_{name}').reset_index())
        out = out.merge(counts, on=group_cols, how='left')
        out[f'n_{name}'] = out[f'n_{name}'].fillna(0).astype(int)
        out[f'prop_{name}'] = out[f'n_{name}'] / out['n_total']
    return out


def plot_criterion(trials, *, criterion=CRITERION, mouse_colours=DEFAULT_MOUSE_COLOURS,
                   mouse_col='mouseID', day_col='training_day', ax=None):
    """Overlay each mouse's success rate per day with a dashed criterion line."""
    summary = make_outcome_df(trials, {'success': ['Success']}, group_cols=(mouse_col, day_col))
    if ax is None:
        fig, ax = plt.subplots(figsize=(7.5, 5))
    else:
        fig = ax.figure
    for mouse in [m for m in mouse_colours if m in set(trials[mouse_col])]:
        s = summary[summary[mouse_col] == mouse].sort_values(day_col)
        ax.plot(s[day_col], s['prop_success'], marker='o', linewidth=2,
                color=mouse_colours[mouse], label=mouse)
    ax.axhline(criterion, linestyle='--', color='crimson', linewidth=1.5,
               label=f'{criterion:.0%} criterion')
    ax.set_xlabel('Training day'); ax.set_ylabel('Success rate'); ax.set_ylim(0, 1)
    ax.set_xticks(sorted(trials[day_col].unique()))
    ax.set_title('Early-training success rate vs 65% criterion')
    ax.legend(fontsize=8)
    return fig, ax, summary


def last_n_days_summary(trials, *, n=2, criterion=CRITERION,
                        mouse_col='mouseID', day_col='training_day'):
    """Per-mouse success rate over the last n days, and whether it clears the criterion."""
    summary = make_outcome_df(trials, {'success': ['Success']}, group_cols=(mouse_col, day_col))
    rows = []
    for mouse, g in summary.groupby(mouse_col):
        last = g.sort_values(day_col).tail(n)
        rows.append({
            'mouseID': mouse,
            'last_days': last[day_col].tolist(),
            'last_rates': [round(r, 3) for r in last['prop_success']],
            f'mean_last{n}': round(last['prop_success'].mean(), 3),
            'every_day_>=criterion': bool((last['prop_success'] >= criterion).all()),
            'mean_>=criterion': bool(last['prop_success'].mean() >= criterion),
        })
    return pd.DataFrame(rows)

In [ ]:
# Load only the listed sessions, then filter (reuses the training-spec helpers).
spec = training_spec(CRITERION_DETAIL)
datastructure = qc_datastructure(
    root=ROOT,
    depth=3,                                              # <tree>/<mouseID>/<day>/<session>
    level_names=('tree', 'mouseID', 'day'),
    streams=('events', 'nosepoke', 'soundcard', 'session_settings'),
    include=training_session_names(spec),
)
result = datastructure.load()
trials_filtered = filter_trials(result['trials'], spec)
trials_filtered

In [ ]:
# Figure + the "did she reach 65%?" table.
fig, ax, summary = plot_criterion(trials_filtered)
plt.show()
last_n_days_summary(trials_filtered, n=2)